In [ ]:
import os
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set up the folder path in Google Drive
folder_path = r'/content/drive/My Drive/Factordata'

# Function to inspect columns of Excel files
def inspect_excel_file_columns(file_path):
    try:
        # Load the specified sheet from the Excel file
        df = pd.read_excel(file_path, sheet_name="FactorOutput")
        print(f"File: {file_path}")
        print("Columns in the FactorOutput sheet:")
        print(df.columns.tolist())  # Print the actual column names
        print("-" * 40)
        return df.columns.tolist()  # Return column names for further inspection
    except Exception as e:
        print(f"Error inspecting {file_path}: {e}")
        return None

# Inspect columns in all files in the specified folder
print("Inspecting columns in each file...\n")
for filename in os.listdir(folder_path):
    if filename.endswith(".xlsx"):
        file_path = os.path.join(folder_path, filename)
        inspect_excel_file_columns(file_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inspecting columns in each file...

File: /content/drive/My Drive/Factordata/Pictet.xlsx
Columns in the FactorOutput sheet:
['Date', 'Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
----------------------------------------
File: /content/drive/My Drive/Factordata/creditsuisse.xlsx
Columns in the FactorOutput sheet:
['Date', 'Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
----------------------------------------
File: /content/drive/My Drive/Factordata/3645.xlsx
Columns in the FactorOutput sheet:
['Date', 'Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
----------------------------------------
File: /content/drive/My Drive/Factordata/Finreon.xlsx
Columns in the FactorOutput sheet:
['Date', 'Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatili

In [ ]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

# Function to process one measure
def process_one_measure(df, measure):
    """Process data for one measure only."""
    df[f'{measure}_Rolling_Mean'] = df[measure].rolling(window=12, min_periods=1).mean()
    df[f'{measure}_Rolling_Std'] = df[measure].rolling(window=12, min_periods=1).std()
    df[f'{measure}_Dynamic_Outlier'] = (
        (df[measure] - df[f'{measure}_Rolling_Mean']).abs() > 1.5 * df[f'{measure}_Rolling_Std']
    )
    return df

# Function to process a single file for a specific measure
def process_file(file_path, measure):
    try:
        df = pd.read_excel(file_path, sheet_name="FactorOutput")
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
        else:
            raise ValueError("The 'Date' column is missing in the file.")

        # Process one measure
        df = process_one_measure(df, measure)
        df['Source_File'] = os.path.basename(file_path)
        return df
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

# List of factors to process
factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']

# Process each factor separately
for measure in factors:
    print(f"Processing measure: {measure}")
    processed_data = []

    # Process all files for the current measure
    for filename in os.listdir(folder_path):
        if filename.endswith(".xlsx"):
            file_path = os.path.join(folder_path, filename)
            processed_df = process_file(file_path, measure)
            if processed_df is not None:
                processed_data.append(processed_df)

    if processed_data:
        combined_data = pd.concat(processed_data, ignore_index=True)
        combined_data['Date'] = pd.to_datetime(combined_data['Date'])
        combined_data.set_index('Date', inplace=True)

        # Ensure unique indices in combined_data
        if combined_data.index.duplicated().any():
            print(f"Found {combined_data.index.duplicated().sum()} duplicate indices. Dropping duplicates...")
            combined_data = combined_data[~combined_data.index.duplicated(keep='first')]

        # Prepare features and labels for supervised learning
        features = combined_data[[measure, f'{measure}_Rolling_Mean', f'{measure}_Rolling_Std']].dropna()
        features['Anomaly_Label'] = combined_data[f'{measure}_Dynamic_Outlier'].astype(int)

        X = features.drop(columns=['Anomaly_Label'])
        y = features['Anomaly_Label']

        # Split data into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

        # Balance training data with SMOTE
        smote = SMOTE(random_state=42, k_neighbors=1)
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

        # Initialize classifiers
        classifiers = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
            'SVM': SVC(probability=True, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42)
        }

        # Train and evaluate models
        results = []
        for name, clf in classifiers.items():
            clf.fit(X_resampled, y_resampled)
            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_proba)
            report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
            results.append({
                'Model': name,
                'Accuracy': report['accuracy'],
                'Precision': report['1']['precision'],
                'Recall': report['1']['recall'],
                'F1-Score': report['1']['f1-score'],
                'ROC-AUC': roc_auc
            })

        # Display results
        results_df = pd.DataFrame(results)
        print(f"Results for measure: {measure}")
        print(results_df)

        # Save results
        results_path = os.path.join(folder_path, f"Model_Comparison_{measure.replace('.', '_')}.xlsx")
        results_df.to_excel(results_path, index=False)
        print(f"Results for {measure} saved to {results_path}")

    else:
        print(f"No data processed for measure: {measure}. Please check the input files.")


Processing measure: Alpha..annualisiert.
Error processing file /content/drive/My Drive/Factordata/Model_Comparison.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Model_Comparison_Updated.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Model_Comparison_All_Factors.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Model_Comparison_Alpha__annualisiert_.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Model_Comparison_Value_Growth.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Model_Comparison_Small_Large.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/My Drive/Factordata/Model_Comparison_Momentum.xlsx: Worksheet named 'FactorOutput' not found
Error processing file /content/drive/